# 01 — Supervisor Architecture

*Level 6 — Multi-Agent RAG*

## Objective
Have a supervisor decide which specialized agent(s) — not just one, unlike Level 3's single-choice router — a business research task should go to, over real data: SEC 10-K filing excerpts (`retrieval-agent`, `research-agent`, `graph-agent`) and the Sakila DVD rental database (`sql-agent`).


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "supervisor"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from multiagent_common.dataset import prepare
from multiagent_common.loader import load_agent_class
from multiagent_common.retrieval import DenseRetriever
from supervisor import Supervisor

data = prepare()
print(f"corpus={len(data.corpus)} filing excerpts across {data.metadata['n_tickers']} companies")

corpus_texts = {cid: c["text"] for cid, c in data.corpus.items()}
retriever = DenseRetriever.from_corpus(corpus_texts)

RetrievalAgent = load_agent_class("retrieval-agent", "RetrievalAgent")
SqlAgent = load_agent_class("sql-agent", "SqlAgent")

agents = {
    "retrieval-agent": RetrievalAgent(retriever, data.corpus),
    "sql-agent": SqlAgent(),
}
supervisor = Supervisor(agents)


corpus=200 filing excerpts across 67 companies


## Route a mix of real tasks


In [3]:
tasks = [
    list(data.questions.values())[0]["question"],
    "How many films are in the database?",
    "What is the total number of customers?",
]
for task in tasks:
    route = supervisor.route(task)
    print(f"{route}  <-  {task[:70]}")


['retrieval-agent', 'graph-agent']  <-  How much were the company's debt obligations as of December 31, 2023?


['sql-agent']  <-  How many films are in the database?


['retrieval-agent', 'sql-agent']  <-  What is the total number of customers?


## Delegate and inspect the real result


In [4]:
task = list(data.questions.values())[0]["question"]
results = supervisor.delegate(task)
for name, result in results.items():
    print(f"[{name}] success={result.success}")
    print(f"  {result.output[:200]}")


[retrieval-agent] success=True
  The company's debt obligations as of December 31, 2023, totaled $2,299,887 thousand.


## What I observed

A financial filing question routes cleanly to `retrieval-agent`; a database-shaped question routes to `sql-agent` — the supervisor is choosing between fundamentally different *kinds* of data sources, not just different phrasings of the same question. Unlike Level 3's router, nothing here restricts the supervisor to exactly one choice — a task genuinely needing both document and graph context gets routed to more than one agent at once.

## Next

[02 — Specialized Agents](./02_specialized_agents.ipynb)
